# Exploitation notebook (no video recording because of OS)

In [1]:
import gym
import highway_env
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO

# Evaluation
from highway_env.envs.common.evaluate import PrintMetrics
import random

# Visualisation   
from ipywidgets import interact, widgets
import glob
import numpy as np

ABS_PATH = 'C:/Users/pigo/Desktop/HighwayDRL'

### Choose the environment to exploit the model in

In [2]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('dm-env-v0', 'acc-dm-env-v0', 'jam-dm-env-v0'…

<function __main__.f(input_env)>

### Choose the model

In [3]:
model_id = widgets.Text()
def m(input_model):
    model_id.value = str(input_model)
interact(m, input_model=glob.glob(f"{ABS_PATH}/training_output/models/*.zip"))


interactive(children=(Dropdown(description='input_model', options=('C:/Users/pigo/Desktop/HighwayDRL/training_…

<function __main__.m(input_model)>

In [4]:
explicit_env = widgets.Checkbox(
    value=False,
    description='explicit env?',
    disabled=False,
    indent=False
)
display(explicit_env)

Checkbox(value=False, description='explicit env?', indent=False)

In [5]:
recurrent = widgets.Checkbox(
    value=False,
    description='recurrent?',
    disabled=False,
    indent=False
)
display(recurrent)

Checkbox(value=False, description='recurrent?', indent=False)

In [6]:
random_agent = widgets.Checkbox(
    value=False,
    description='random behaviour?',
    disabled=False,
    indent=False
)
display(random_agent)

Checkbox(value=False, description='random behaviour?', indent=False)

In [7]:
manual = widgets.Checkbox(
    value=False,
    description='manual control?',
    disabled=False,
    indent=False
)
display(manual)

Checkbox(value=False, description='manual control?', indent=False)

In [8]:
render = widgets.Checkbox(
    value=False,
    description='render scene?',
    disabled=False,
    indent=False
)
display(render)

Checkbox(value=False, description='render scene?', indent=False)

In [9]:
# Number of exploitation episodes
episode_num = 60
metricObj = PrintMetrics()
# metric counters
left_lane_change, right_lane_change = None, None

expl_envs = ['dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','multipleO-dm-env-v0']

if(explicit_env.value):
    env = gym.make(env_id.value)
    random_env = env_id.value
    env.configure({
        "screen_width":1500,
        "screen_height":250,
        "manual-control": manual.value
    })
if(not manual.value and not random_agent.value):
    if(recurrent.value):
        model = RecurrentPPO.load(model_id.value)
        print('using recurrent')
    else:
        model = PPO.load(model_id.value)
        print('using standard PPO')


for episode in range(episode_num):
    if(not explicit_env.value):
        random_env = random.choice(expl_envs)
        env = gym.make(random_env)
        env.configure({
            "screen_width":1500,
            "screen_height":250,
            "manual-control": manual.value
        })
        
    obs, done = env.reset(), False
    if(recurrent.value):
        lstm_states = None
        num_envs = 1
        # Episode start signals are used to reset the lstm states
        episode_starts = np.ones((num_envs,), dtype=bool)

    # Reset metric counters at the beginning of each episode
    decision_change_num, left_lane_change_num, right_lane_change_num = 0, 0, 0
    accelerations, decelerations, speeds = [], [], []
   
    while not done:
        if(random_agent.value):
            action = env.random_action()
            obs, reward, done, info = env.step(action)
        elif(manual.value):
            env.step(env.action_space.sample())
        else:
            if(recurrent.value):
                action, lstm_states = model.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
            else:
                action, _states = model.predict(obs, deterministic=True)
            obs, reward, done, info = env.step(action)
            if(recurrent.value):
                episode_starts = done
            
        if(render.value):
            env.render()

        # Update metric counters
        decision_change_num += env.DECISION_CHANGE
        speeds.append(env.controlled_vehicles[0].speed)
        if env.controlled_vehicles[0].throttle > 0:
            accelerations.append(env.controlled_vehicles[0].throttle)
        else:
            decelerations.append(env.controlled_vehicles[0].throttle)

        if env.LAST_LANE_IDX != env.controlled_vehicles[0].lane_index[2]:
            if env.LAST_LANE_IDX > env.controlled_vehicles[0].lane_index[2] and env.LAST_LANE_IDX != 1000:
                left_lane_change_num += 1
            else:
                right_lane_change_num += 1
            env.LAST_LANE_IDX = env.controlled_vehicles[0].lane_index[2]

    episode_duration = (env.steps/env.config["policy_frequency"]*env.config["simulation_frequency"])/30
    decision_change_rate = decision_change_num / episode_duration
    left_lane_change_rate = left_lane_change_num / episode_duration
    right_lane_change_rate = right_lane_change_num / episode_duration
    mean_speed = np.mean(speeds)
    km_travelled = (mean_speed * episode_duration)/1000
    mean_acceleration = np.mean(accelerations)
    mean_deceleration = np.mean(decelerations)
    metricObj.printEpisode(random_env, km_travelled, decision_change_num, decision_change_rate, left_lane_change_num, left_lane_change_rate,\
        right_lane_change_num, right_lane_change_rate, mean_speed*3.6, mean_acceleration, mean_deceleration, env.controlled_vehicles[0].crashed, episode_duration, episode+1)

env.close()
if random_agent.value:
    csv_id = 'random_agent'
elif manual.value:
    csv_id = 'manual'
else:
    csv_id = model_id.value[model_id.value.find('REC'):model_id.value.find('.zip')] if recurrent.value else model_id.value[model_id.value.find('ppo'):model_id.value.find('.zip')]
    
metricObj.printRecap(f"{ABS_PATH}/exploitation/eval_metrics/", csv_id)

using standard PPO
high speed rew: 0.0, final rew: 0
high speed rew: 0.13199929921681294, final rew: 0
high speed rew: 0.24178444463481605, final rew: 0
high speed rew: 0.306596654835748, final rew: 0
high speed rew: 0.3448588675889937, final rew: 0
high speed rew: 0.36744715643512316, final rew: 0
high speed rew: 0.38078226583631364, final rew: 0
high speed rew: 0.3886547144291692, final rew: 0
high speed rew: 0.3933022538667994, final rew: 0
high speed rew: 0.3960459520401907, final rew: 0
high speed rew: 0.3976657079922788, final rew: 0
high speed rew: 0.39862193902737225, final rew: 0
high speed rew: 0.39918645480599846, final rew: 0
high speed rew: 0.3995197195219731, final rew: 0
high speed rew: 0.39971646401543004, final rew: 0
high speed rew: 0.3998326131120786, final rew: 0
high speed rew: 0.3999011823127475, final rew: 0
high speed rew: 0.3999416624836324, final rew: 0
high speed rew: 0.3999655601551652, final rew: 0
high speed rew: 0.39997966826518955, final rew: 0
high spee

KeyboardInterrupt: 